# Preprocessing Data — Online Retail Dataset

**Mata Kuliah**: Data Mining (21TIF604)  
**Universitas Islam Nahdlatul Ulama Jepara**  
**Dosen**: Ir. Adi Sucipto, M.Kom

---

### Big Picture — Apa itu Data Mining & KDD?

**Data Mining** adalah proses menemukan pola, tren, dan pengetahuan baru dari data besar — yang sebelumnya tersembunyi. Proses ini merupakan bagian dari **KDD (Knowledge Discovery in Database)** yang terdiri dari 5 tahap:

1. **Selection** — memilih data yang relevan
2. **Preprocessing** — membersihkan data dari masalah (missing value, duplikat, outlier)
3. **Transformation** — mengubah data ke format yang sesuai untuk algoritma
4. **Data Mining** — menjalankan algoritma untuk menemukan pola
5. **Interpretation/Evaluation** — menafsirkan dan mengevaluasi hasil

Di proyek ini, kita menjalankan **seluruh siklus KDD** menggunakan 3 metode:
- **Klasifikasi (C4.5)** → memprediksi: "Pelanggan ini loyal atau tidak?"
- **Clustering (K-Means)** → mengelompokkan: "Segmen mana pelanggan ini masuk?"
- **Association Rules (Apriori)** → menemukan pola: "Produk apa yang sering dibeli bersamaan?"

---

### Tujuan Video Ini:
1. Mengambil dataset dari Kaggle menggunakan `kagglehub`
2. Memahami struktur dan karakteristik data
3. Mengidentifikasi masalah data (missing value, duplikat, outlier)
4. Melakukan preprocessing lengkap step-by-step
5. Menyimpan data bersih untuk digunakan di video selanjutnya

---

### Konteks Renstra (Kemanfaatan):
Dataset **Online Retail** berisi transaksi retail online real-world dari UK periode 2010-2011. Melalui analisis data mining, kita dapat menghasilkan rekomendasi bisnis yang langsung applicable, seperti: identifikasi pelanggan loyal, segmentasi pelanggan untuk strategi pemasaran yang tepat sasaran, dan penemuan pola pembelian untuk bundling produk.

---

### Sitasi Dataset:
> Chen, D. (2012). Online Retail [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D  
> *(Dataset diakses melalui mirror Kaggle: lakshmi25npathi/online-retail-dataset)*

In [ ]:
# ============================================================
# STEP 1: INSTALL LIBRARY YANG BELUM ADA DI GOOGLE COLAB
# ============================================================
# Google Colab sudah punya: pandas, numpy, matplotlib, seaborn, scikit-learn
# Yang perlu diinstall tambahan: kagglehub, mlxtend, openpyxl
# ============================================================

!pip install kagglehub mlxtend openpyxl -q

print("✅ Library tambahan berhasil diinstall!")

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE & IMPORT LIBRARY
# ============================================================
# Mount Drive agar bisa menyimpan/memuat file dari Google Drive
# File yang disimpan di Drive akan tetap ada meskipun sesi Colab berakhir
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import kagglehub                              # Untuk mengambil dataset dari Kaggle
import pandas as pd                           # Library utama manipulasi data (tabel/dataframe)
import numpy as np                            # Library operasi numerik dan array
import matplotlib.pyplot as plt               # Library membuat grafik/visualisasi
import seaborn as sns                         # Library visualisasi lebih cantik (berbasis matplotlib)
import os                                     # Library operasi sistem (path, file)

# Pengaturan tampilan DataFrame agar lebih mudah dibaca
pd.set_option('display.max_columns', None)    # Tampilkan semua kolom tanpa terpotong
pd.set_option('display.max_rows', 100)        # Tampilkan maksimal 100 baris

# Pengaturan ukuran grafik default
plt.rcParams['figure.figsize'] = (12, 6)      # Lebar 12 inch, tinggi 6 inch

# Folder kerja di Google Drive
DRIVE_FOLDER = '/content/drive/MyDrive/data-mining'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

print("✅ Google Drive ter-mount & semua library berhasil diimport!")
print(f"📁 Folder kerja: {DRIVE_FOLDER}")

## Step 2: Load Dataset dari Kaggle

Dataset: **Online Retail** — sumber asli dari UCI Machine Learning Repository  
> Chen, D. (2012). Online Retail [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D  
> *(Diakses melalui mirror Kaggle: lakshmi25npathi/online-retail-dataset)*

Berisi transaksi retail online dari UK periode 2010-2011.

**Kolom-kolom dataset:**
- `InvoiceNo` — Nomor transaksi (jika diawali 'C' = pembatalan)
- `StockCode` — Kode produk
- `Description` — Nama produk
- `Quantity` — Jumlah item per transaksi
- `InvoiceDate` — Tanggal dan waktu transaksi
- `UnitPrice` — Harga per unit (Poundsterling)
- `CustomerID` — ID pelanggan
- `Country` — Negara pelanggan

In [ ]:
# ============================================================
# LOAD DATASET DARI KAGGLE
# ============================================================
# Langkah 1: Download seluruh dataset dari Kaggle ke lokal Colab
# Langkah 2: Baca file Excel menggunakan pandas
#
# Mengapa dua langkah? Karena kagglehub.load_dataset sering error 404
# untuk file tertentu. Lebih aman download bundle lalu baca manual.
# ============================================================

# Download dataset (menghasilkan path folder tempat file disimpan)
dataset_path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")
print(f"📂 Dataset terdownload di: {dataset_path}")

# Cari file Excel di dalam folder dataset
import glob
xlsx_files = glob.glob(os.path.join(dataset_path, "*.xlsx"))
print(f"📋 File Excel ditemukan: {xlsx_files}")

# Baca file Excel pertama yang ditemukan
if xlsx_files:
    df = pd.read_excel(xlsx_files[0])
    print(f"✅ Dataset berhasil dimuat!")
    print(f"   Jumlah baris: {df.shape[0]:,}")
    print(f"   Jumlah kolom: {df.shape[1]}")
else:
    # Jika tidak ada .xlsx, coba .csv
    csv_files = glob.glob(os.path.join(dataset_path, "*.csv"))
    if csv_files:
        df = pd.read_csv(csv_files[0], encoding='latin1')
        print(f"✅ Dataset (CSV) berhasil dimuat!")
        print(f"   Jumlah baris: {df.shape[0]:,}")
        print(f"   Jumlah kolom: {df.shape[1]}")
    else:
        print("❌ Tidak ada file dataset ditemukan!")

In [ ]:
# ============================================================
# CEK & STANDARISASI NAMA KOLOM
# ============================================================
# Dataset dari Kaggle bisa punya nama kolom yang berbeda-beda:
# misalnya "UnitPrice" vs "unit price" vs "Price" vs " UnitPrice "
# Dataset Online Retail II punya "Invoice" bukan "InvoiceNo"
# Kita standarisasi agar konsisten di seluruh notebook
# ============================================================

print(f"📋 Nama kolom asli dari dataset:")
print(f"   {list(df.columns)}")

# Standarisasi: hapus spasi di awal/akhir, ubah ke format CamelCase standar
df.columns = df.columns.str.strip()  # Hapus spasi berlebih

# Mapping nama kolom yang mungkin berbeda
rename_map = {}
for col in df.columns:
    col_lower = col.lower().replace(' ', '').replace('_', '')
    if col_lower == 'invoiceno' or col_lower == 'invoice' or col_lower == 'invoicenumber' or col_lower == 'invoicenum':
        rename_map[col] = 'InvoiceNo'
    elif col_lower == 'unitprice' or col_lower == 'price':
        rename_map[col] = 'UnitPrice'
    elif col_lower == 'stockcode' or col_lower == 'itemcode':
        rename_map[col] = 'StockCode'
    elif col_lower == 'description' or col_lower == 'itemdescription':
        rename_map[col] = 'Description'
    elif col_lower == 'quantity' or col_lower == 'qty':
        rename_map[col] = 'Quantity'
    elif col_lower == 'invoicedate' or col_lower == 'date':
        rename_map[col] = 'InvoiceDate'
    elif col_lower == 'customerid' or col_lower == 'custid' or col_lower == 'userid':
        rename_map[col] = 'CustomerID'
    elif col_lower == 'country':
        rename_map[col] = 'Country'

if rename_map:
    df.rename(columns=rename_map, inplace=True)
    print(f"\n🔄 Kolom yang di-rename: {rename_map}")

print(f"\n📋 Nama kolom setelah standarisasi:")
print(f"   {list(df.columns)}")
print(f"\n✅ Nama kolom sudah distandarisasi!")

## Step 3: Eksplorasi Data Awal (Initial Data Exploration)

Sebelum preprocessing, kita harus **mengenal data terlebih dahulu**.  
Ini adalah tahap penting dalam proses KDD (Knowledge Discovery in Database).

In [ ]:
# ============================================================
# MELIHAT 5 BARIS PERTAMA DATASET
# ============================================================
# Tujuan: Memahami bentuk dan format data secara visual
# ============================================================

print("📋 5 Baris Pertama Dataset:")
print("=" * 80)
df.head()

In [ ]:
# ============================================================
# INFORMASI UMUM DATASET
# ============================================================
# df.info() menampilkan: nama kolom, jumlah non-null, tipe data, penggunaan memori
# ============================================================

print("📋 Informasi Struktur Dataset:")
print("=" * 80)
df.info()

In [ ]:
# ============================================================
# STATISTIK DESKRIPTIF KOLOM NUMERIK
# ============================================================
# Menampilkan: count, mean, std, min, 25%/50%/75% (kuartil), max
# ============================================================

print("📊 Statistik Deskriptif Kolom Numerik:")
print("=" * 80)
df.describe()

In [ ]:
# ============================================================
# STATISTIK DESKRIPTIF KOLOM KATEGORIK (TEKS)
# ============================================================
# Menampilkan: count, unique, top (nilai terbanyak), freq (frekuensi top)
# ============================================================

print("📊 Statistik Deskriptif Kolom Kategorik:")
print("=" * 80)
df.describe(include='object')

## Step 4: Identifikasi Masalah Data

Dalam data mining, data mentah hampir selalu memiliki masalah. Kita perlu mengidentifikasi:
- **Missing value** — data kosong/null
- **Data duplikat** — baris identik
- **Nilai anomali/outlier** — nilai yang tidak wajar
- **Tipe data salah** — kolom belum sesuai tipe yang seharusnya

In [ ]:
# ============================================================
# CEK MISSING VALUE (NILAI KOSONG) PADA SETIAP KOLOM
# ============================================================
# Missing value = data yang tidak terisi/kosong (NaN/Null)
# Harus ditangani karena bisa membuat hasil analisis tidak valid
# ============================================================

missing = pd.DataFrame({
    'Jumlah Missing': df.isnull().sum(),
    'Persentase (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

# Tampilkan hanya kolom yang punya missing value
print("🔍 Missing Value per Kolom:")
print("=" * 80)
print(missing[missing['Jumlah Missing'] > 0])
print(f"\n⚠️ Total baris dengan missing value: {df.isnull().any(axis=1).sum():,} dari {len(df):,} baris")
print(f"⚠️ Persentase baris bermasalah: {(df.isnull().any(axis=1).sum() / len(df) * 100):.2f}%")

In [ ]:
# ============================================================
# VISUALISASI MISSING VALUE
# ============================================================

cols_missing = missing[missing['Jumlah Missing'] > 0].index.tolist()

if len(cols_missing) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(cols_missing, missing.loc[cols_missing, 'Persentase (%)'], color='#e74c3c')
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title('Persentase Missing Value per Kolom', fontsize=14, fontweight='bold')
    ax.set_ylabel('Persentase (%)', fontsize=12)
    ax.set_xlabel('Kolom', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("✅ Tidak ada missing value!")

In [ ]:
# ============================================================
# CEK DATA DUPLIKAT
# ============================================================
# Data duplikat = baris yang identik sepenuhnya dengan baris lain
# Membuat analisis bias, harus dihapus
# ============================================================

jumlah_duplikat = df.duplicated().sum()
persentase_duplikat = (jumlah_duplikat / len(df)) * 100

print(f"🔍 Jumlah baris duplikat: {jumlah_duplikat:,} dari {len(df):,} baris")
print(f"📊 Persentase duplikat: {persentase_duplikat:.2f}%")

# Tampilkan beberapa contoh baris duplikat
if jumlah_duplikat > 0:
    print(f"\n📋 Contoh baris duplikat:")
    print(df[df.duplicated(keep='first')].head(5))

In [ ]:
# ============================================================
# CEK NILAI ANOMALI PADA KOLOM NUMERIK
# ============================================================
# Quantity negatif = pembatalan transaksi (InvoiceNo diawali 'C')
# UnitPrice ≤ 0 = tidak valid untuk transaksi penjualan
# ============================================================

print("🔍 Cek Nilai Anomali pada Kolom Numerik:")
print("=" * 80)

# Cek Quantity negatif
qty_negatif = df[df['Quantity'] < 0]
print(f"⚠️ Jumlah baris Quantity NEGATIF: {len(qty_negatif):,}")
print(f"   (Kemungkinan transaksi pembatalan, InvoiceNo diawali 'C')")

# Cek UnitPrice ≤ 0
price_anomali = df[df['UnitPrice'] <= 0]
print(f"⚠️ Jumlah baris UnitPrice ≤ 0: {len(price_anomali):,}")

# Distribusi Quantity
print(f"\n📊 Distribusi Quantity:")
print(f"   Min: {df['Quantity'].min()}, Max: {df['Quantity'].max()}, Median: {df['Quantity'].median()}")

# Distribusi UnitPrice
print(f"📊 Distribusi UnitPrice:")
print(f"   Min: {df['UnitPrice'].min()}, Max: {df['UnitPrice'].max()}, Median: {df['UnitPrice'].median()}")

In [ ]:
# ============================================================
# VISUALISASI OUTLIER DENGAN BOXPLOT (SEBELUM CLEANING)
# ============================================================
# Boxplot menampilkan: kuartil bawah, median, kuartil atas, dan outlier (titik di luar whisker)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot Quantity
sns.boxplot(x=df['Quantity'], ax=axes[0], color='#3498db')
axes[0].set_title('Boxplot Quantity (SEBELUM Cleaning)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Quantity', fontsize=11)

# Boxplot UnitPrice
sns.boxplot(x=df['UnitPrice'], ax=axes[1], color='#2ecc71')
axes[1].set_title('Boxplot UnitPrice (SEBELUM Cleaning)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('UnitPrice (Poundsterling)', fontsize=11)

plt.tight_layout()
plt.show()

print("💡 Terlihat banyak outlier ekstrem pada kedua kolom. Kita akan menanganinya di tahap cleaning.")

## Step 5: Data Cleaning (Pembersihan Data)

Sekarang kita sudah tahu masalah-masalah dalam data. Tahap cleaning mencakup:
1. Hapus missing value pada kolom penting
2. Hapus data duplikat
3. Hapus transaksi pembatalan (Quantity negatif)
4. Hapus baris dengan harga tidak valid (≤ 0)
5. Hapus outlier ekstrem (metode IQR)
6. Perbaiki tipe data yang salah

**Setiap langkah didokumentasi jumlah sebelum/sesudah** — transparan & akuntabeL

In [ ]:
# ============================================================
# CATAT JUMLAH DATA AWAL SEBELUM CLEANING
# ============================================================
# Ini penting untuk dokumentasi: berapa banyak data yang dibersihkan
# ============================================================

jumlah_awal = len(df)
print(f"📊 Jumlah data AWAL: {jumlah_awal:,} baris")

# Buat salinan dataframe agar data asli tetap tersimpan
# Praktik baik: kita bisa membandingkan sebelum/sesudah cleaning
df_clean = df.copy()
print("✅ Salinan dataframe dibuat. Data asli tetap aman.")

In [ ]:
# ============================================================
# CLEANING 1: HAPUS MISSING VALUE pada CustomerID & Description
# ============================================================
# CustomerID wajib untuk analisis pelanggan (klasifikasi & clustering)
# Description wajib untuk association rules (pola produk)
# ============================================================

sebelum = len(df_clean)

# Hapus baris yang CustomerID-nya kosong (NaN)
df_clean = df_clean.dropna(subset=['CustomerID'])

# Hapus baris yang Description-nya kosong (NaN)
df_clean = df_clean.dropna(subset=['Description'])

sesudah = len(df_clean)
dihapus = sebelum - sesudah

print(f"🧹 Cleaning 1 - Hapus Missing Value:")
print(f"   Sebelum: {sebelum:,} baris")
print(f"   Sesudah: {sesudah:,} baris")
print(f"   Dihapus: {dihapus:,} baris ({(dihapus/sebelum)*100:.2f}%)")

In [ ]:
# ============================================================
# CLEANING 2: HAPUS DATA DUPLIKAT
# ============================================================
# keep='first' = simpan kemunculan pertama, hapus sisanya
# Duplikat membuat analisis bias (satu transaksi dihitung berkali-kali)
# ============================================================

sebelum = len(df_clean)

df_clean = df_clean.drop_duplicates(keep='first')

sesudah = len(df_clean)
dihapus = sebelum - sesudah

print(f"🧹 Cleaning 2 - Hapus Duplikat:")
print(f"   Sebelum: {sebelum:,} baris")
print(f"   Sesudah: {sesudah:,} baris")
print(f"   Dihapus: {dihapus:,} baris ({(dihapus/sebelum)*100:.2f}%)")

In [ ]:
# ============================================================
# CLEANING 3: HAPUS TRANSAKSI PEMBATALAN (Quantity ≤ 0)
# ============================================================
# Quantity negatif = pembatalan transaksi (InvoiceNo diawali 'C')
# Untuk analisis data mining, kita hanya butuh transaksi aktual
# ============================================================

sebelum = len(df_clean)

# Hapus semua baris yang Quantity-nya negatif atau nol
df_clean = df_clean[df_clean['Quantity'] > 0]

sesudah = len(df_clean)
dihapus = sebelum - sesudah

print(f"🧹 Cleaning 3 - Hapus Transaksi Pembatalan:")
print(f"   Sebelum: {sebelum:,} baris")
print(f"   Sesudah: {sesudah:,} baris")
print(f"   Dihapus: {dihapus:,} baris ({(dihapus/sebelum)*100:.2f}%)")

In [ ]:
# ============================================================
# CLEANING 4: HAPUS HARGA TIDAK VALID (UnitPrice ≤ 0)
# ============================================================
# UnitPrice ≤ 0 tidak masuk akal untuk transaksi penjualan
# Bisa jadi data error, gratis, atau biaya tambahan (bukan penjualan)
# ============================================================

sebelum = len(df_clean)

# Hapus baris yang UnitPrice-nya kurang dari atau sama dengan 0
df_clean = df_clean[df_clean['UnitPrice'] > 0]

sesudah = len(df_clean)
dihapus = sebelum - sesudah

print(f"🧹 Cleaning 4 - Hapus Harga Tidak Valid:")
print(f"   Sebelum: {sebelum:,} baris")
print(f"   Sesudah: {sesudah:,} baris")
print(f"   Dihapus: {dihapus:,} baris ({(dihapus/sebelum)*100:.2f}%)")

In [ ]:
# ============================================================
# CLEANING 5: HAPUS OUTLIER EKSTREM DENGAN METODE IQR
# ============================================================
# IQR = Interquartile Range (Rentang Kuartil)
# Rumus outlier:
#   Batas bawah = Q1 - 1.5 * IQR
#   Batas atas  = Q3 + 1.5 * IQR
# Di mana: Q1 = kuartil 25%, Q3 = kuartil 75%, IQR = Q3 - Q1
# ============================================================

sebelum = len(df_clean)

# --- Hitung batas outlier untuk Quantity ---
Q1_qty = df_clean['Quantity'].quantile(0.25)   # Kuartil pertama (25%)
Q3_qty = df_clean['Quantity'].quantile(0.75)   # Kuartil ketiga (75%)
IQR_qty = Q3_qty - Q1_qty                      # Rentang kuartil
batas_bawah_qty = Q1_qty - 1.5 * IQR_qty      # Batas bawah outlier
batas_atas_qty = Q3_qty + 1.5 * IQR_qty        # Batas atas outlier

print(f"📏 Batas outlier Quantity:")
print(f"   Q1={Q1_qty}, Q3={Q3_qty}, IQR={IQR_qty}")
print(f"   Batas bawah={batas_bawah_qty}, Batas atas={batas_atas_qty}")

# --- Hitung batas outlier untuk UnitPrice ---
Q1_prc = df_clean['UnitPrice'].quantile(0.25)   # Kuartil pertama (25%)
Q3_prc = df_clean['UnitPrice'].quantile(0.75)   # Kuartil ketiga (75%)
IQR_prc = Q3_prc - Q1_prc                       # Rentang kuartil
batas_bawah_prc = Q1_prc - 1.5 * IQR_prc        # Batas bawah outlier
batas_atas_prc = Q3_prc + 1.5 * IQR_prc         # Batas atas outlier

print(f"\n📏 Batas outlier UnitPrice:")
print(f"   Q1={Q1_prc}, Q3={Q3_prc}, IQR={IQR_prc}")
print(f"   Batas bawah={batas_bawah_prc}, Batas atas={batas_atas_prc}")

# Filter: hanya simpan baris yang TIDAK outlier pada kedua kolom
df_clean = df_clean[
    (df_clean['Quantity'] >= batas_bawah_qty) & (df_clean['Quantity'] <= batas_atas_qty) &
    (df_clean['UnitPrice'] >= batas_bawah_prc) & (df_clean['UnitPrice'] <= batas_atas_prc)
]

sesudah = len(df_clean)
dihapus = sebelum - sesudah

print(f"\n🧹 Cleaning 5 - Hapus Outlier Ekstrem (IQR):")
print(f"   Sebelum: {sebelum:,} baris")
print(f"   Sesudah: {sesudah:,} baris")
print(f"   Dihapus: {dihapus:,} baris ({(dihapus/sebelum)*100:.2f}%)")

In [ ]:
# ============================================================
# CLEANING 6: PERBAIKI TIPE DATA & TAMBAH KOLOM BARU
# ============================================================
# CustomerID seharusnya integer (bukan float)
# InvoiceDate seharusnya datetime (bukan string/object)
# TotalAmount = Quantity × UnitPrice (total belanja per baris)
# ============================================================

# Ubah CustomerID dari float ke integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# Ubah InvoiceDate dari string ke datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Tambah kolom TotalAmount: total belanja per baris transaksi
# Berguna untuk analisis clustering (segmentasi pelanggan)
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print("✅ Tipe data diperbaiki:")
print(f"   CustomerID → int (sebelumnya float)")
print(f"   InvoiceDate → datetime (sebelumnya object/string)")
print(f"   TotalAmount (kolom baru) = Quantity × UnitPrice")
print(f"\n📋 Tipe data setelah perbaikan:")
print(df_clean.dtypes)

## Step 6: Verifikasi Hasil Cleaning

Kita membandingkan kondisi data sebelum dan sesudah cleaning untuk memastikan semua masalah sudah tertangani.

In [ ]:
# ============================================================
# RINGKASAN HASIL PREPROCESSING
# ============================================================

print("=" * 60)
print("📊 RINGKASAN PREPROCESSING")
print("=" * 60)
print(f"Data AWAL   : {jumlah_awal:,} baris")
print(f"Data BERSIH : {len(df_clean):,} baris")
print(f"Dihapus     : {jumlah_awal - len(df_clean):,} baris ({(jumlah_awal - len(df_clean))/jumlah_awal*100:.2f}%)")
print()
print(f"✅ Missing value tersisa : {df_clean.isnull().sum().sum()} (harusnya 0)")
print(f"✅ Duplikat tersisa      : {df_clean.duplicated().sum()} (harusnya 0)")
print(f"✅ Quantity negatif      : {len(df_clean[df_clean['Quantity'] <= 0])} (harusnya 0)")
print(f"✅ UnitPrice ≤ 0         : {len(df_clean[df_clean['UnitPrice'] <= 0])} (harusnya 0)")

In [ ]:
# ============================================================
# BOXPLOT SESUDAH CLEANING (bandingkan dengan sebelum)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df_clean['Quantity'], ax=axes[0], color='#3498db')
axes[0].set_title('Boxplot Quantity (SESUDAH Cleaning)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Quantity', fontsize=11)

sns.boxplot(x=df_clean['UnitPrice'], ax=axes[1], color='#2ecc71')
axes[1].set_title('Boxplot UnitPrice (SESUDAH Cleaning)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('UnitPrice (Poundsterling)', fontsize=11)

plt.tight_layout()
plt.show()

print("💡 Outlier berkurang drastis setelah cleaning. Data sekarang lebih bersih dan valid.")

## Step 7: Simpan Data Bersih

Data bersih ini akan digunakan di video 2-5 (klasifikasi, clustering, association rules, evaluasi).

In [ ]:
# ============================================================
# SIMPAN DATA BERSIH KE GOOGLE DRIVE
# ============================================================
# Data bersih ini akan digunakan di tahap berikutnya (klasifikasi, clustering, dll)
# Disimpan di Google Drive agar tetap ada meskipun sesi Colab berakhir
# ============================================================

output_path = os.path.join(DRIVE_FOLDER, 'online_retail_clean.csv')
df_clean.to_csv(output_path, index=False)

print(f"✅ Data bersih disimpan ke: {output_path}")
print(f"   Jumlah baris: {len(df_clean):,}")
print(f"   Jumlah kolom: {df_clean.shape[1]}")
print(f"   Ukuran file: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")